# 01 Data Cleaning

This notebook performs basic data cleaning for the Taipei housing transaction dataset.

## 1. Load Data

In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/final_data.csv")
df.columns = df.columns.str.lower()

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 41 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   _id           919 non-null    int64  
 1   _importdate   919 non-null    object 
 2   case_t        919 non-null    object 
 3   district      919 non-null    object 
 4   case_f        919 non-null    object 
 5   build_name    174 non-null    object 
 6   location      919 non-null    object 
 7   landa         919 non-null    float64
 8   landa_z       0 non-null      float64
 9   sdate         919 non-null    int64  
 10  scnt          919 non-null    object 
 11  trans_target  174 non-null    object 
 12  sbuild        861 non-null    object 
 13  tbuild        524 non-null    float64
 14  buitype       841 non-null    object 
 15  pbuild        821 non-null    object 
 16  mbuild        179 non-null    object 
 17  fdate         5 non-null      float64
 18  farea         919 non-null    

## 2. Basic Cleaning

In [3]:
# Keep only housing transaction data in year 113
df = df[df["case_t"].isin(["買賣", "預售屋"])]
df = df[df["sdate"].astype(str).str.startswith("113")]

In [4]:
# Drop columns that should not be used for prediction
drop_cols = [
    "_id",
    "_importdate",
    "location",
    "build_name",
    "rmnote",
    "sdate",
    "uprice",      
    "pprice",      
    "longitude",
    "latitude",

    # rental-only columns
    "rent_type",
    "rent_period",
    "rent_service"
]

df = df.drop(columns=drop_cols, errors="ignore")

In [5]:
# Convert numeric columns
numeric_cols = [
    "landa", "farea", "build_r", "build_l", "build_b",
    "parea", "mb_area", "sb_area", "pu_area",
    "tbuild", "fdate", "tprice", "metro"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [6]:
# Remove rows without target or basic area information
df = df.dropna(subset=["tprice", "farea"])
df = df[(df["tprice"] > 0) & (df["farea"] > 0)]

In [7]:
# Drop columns with more than 50% missing values
missing_ratio = df.isna().mean()
high_missing_cols = missing_ratio[missing_ratio > 0.5].index.tolist()

df = df.drop(columns=high_missing_cols)

In [8]:
# Remove duplicate rows
df = df.drop_duplicates()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 490 entries, 1 to 917
Data columns (total 22 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   case_t        490 non-null    object 
 1   district      490 non-null    object 
 2   case_f        490 non-null    object 
 3   landa         490 non-null    float64
 4   scnt          490 non-null    object 
 5   sbuild        490 non-null    object 
 6   buitype       472 non-null    object 
 7   pbuild        476 non-null    object 
 8   farea         490 non-null    float64
 9   build_r       422 non-null    float64
 10  build_l       422 non-null    float64
 11  build_b       422 non-null    float64
 12  build_p       490 non-null    object 
 13  rule          318 non-null    object 
 14  tprice        490 non-null    int64  
 15  upnote        490 non-null    object 
 16  parktype      247 non-null    object 
 17  parea         490 non-null    float64
 18  elevator      318 non-null    objec

## 3. Save Cleaned Data

In [ ]:
df.to_csv("../data/cleaned_data.csv", index=False)